# FitCheck — Resume/JD Fit Classifier Training

Fine-tunes DistilBERT as a 3-class resume<->job-description fit classifier on the
[cnamuangtoun/resume-job-description-fit](https://huggingface.co/datasets/cnamuangtoun/resume-job-description-fit)
dataset (8,000 real resume-JD pairs), then exports to ONNX for the FitCheck Streamlit app.

**Before running:** in the Kaggle notebook settings (right sidebar), turn on:
- **Accelerator: GPU T4 x2** (or any GPU)
- **Internet: On** (needed to download the dataset and the base DistilBERT weights)

Training auto-falls-back to CPU if Kaggle assigns a GPU too old for the installed PyTorch
build (e.g. a P100) -- it'll just be slower, not crash.

This notebook writes out the same scripts that live in `training/scripts/` in the FitCheck
repo (kept in sync — copy-pasted verbatim, not reimplemented), so behavior matches what you'd
get running them locally.

Run all cells top to bottom. At the end, `fitcheck_model.zip` in the notebook's Output tab
contains everything to drop into `FitCheck/models/` locally.

In [ ]:
!pip install -q transformers onnxruntime

## 1. Prepare dataset

In [ ]:
%%writefile /kaggle/working/prepare_dataset.py
"""Download the resume-JD fit dataset from Hugging Face and carve out a val split.

Source: cnamuangtoun/resume-job-description-fit (6,241 train / 1,759 test rows,
columns: resume_text, job_description_text, label in {No Fit, Potential Fit, Good Fit}).
The published test.csv is kept untouched as the final holdout; val.csv is a
stratified slice of train.csv so hyperparameters are never tuned against test.

Usage (Kaggle notebook, internet enabled):
    python prepare_dataset.py --output_dir /kaggle/working/data
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import pandas as pd
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split

REPO_ID = "cnamuangtoun/resume-job-description-fit"

# Fixed order (not alphabetical) so the label is ordinal: index reflects fit strength.
LABELS = ["No Fit", "Potential Fit", "Good Fit"]


def download_csv(filename: str, retries: int = 5, backoff_seconds: float = 5.0) -> pd.DataFrame:
    """Uses huggingface_hub's resolution API rather than a raw GET against resolve/main/ --
    a raw GET against the CSV URL was seen to hit sustained 504s from Kaggle's network across
    every retry, while hf_hub_download hits HF's actual CDN resolution path and handles its
    own connection retries too."""
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            path = hf_hub_download(repo_id=REPO_ID, filename=filename, repo_type="dataset")
            return pd.read_csv(path)
        except Exception as e:
            last_error = e
            if attempt == retries:
                raise
            wait = backoff_seconds * (2 ** (attempt - 1))
            print(f"download failed ({e}), retrying in {wait:.0f}s ({attempt}/{retries})")
            time.sleep(wait)
    raise last_error  # unreachable, satisfies type checkers


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--output_dir", required=True, type=Path)
    parser.add_argument("--val_size", type=float, default=0.15)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    args.output_dir.mkdir(parents=True, exist_ok=True)

    full_train = download_csv("train.csv")
    test = download_csv("test.csv")
    print(f"downloaded: train={len(full_train)} test={len(test)}")

    assert set(full_train["label"].unique()) <= set(LABELS), "unexpected label values"

    train, val = train_test_split(
        full_train, test_size=args.val_size, random_state=args.seed, stratify=full_train["label"]
    )
    print(f"split: train={len(train)} val={len(val)} test={len(test)}")

    train.to_csv(args.output_dir / "train.csv", index=False)
    val.to_csv(args.output_dir / "val.csv", index=False)
    test.to_csv(args.output_dir / "test.csv", index=False)

    with open(args.output_dir / "labels.json", "w") as f:
        json.dump(LABELS, f, indent=2)
    print(f"labels.json written: {LABELS}")


if __name__ == "__main__":
    main()


In [ ]:
!python /kaggle/working/prepare_dataset.py --output_dir /kaggle/working/data

## 2. Train

In [ ]:
%%writefile /kaggle/working/train_config.yaml
data_dir: /kaggle/working/data   # output of prepare_dataset.py: train.csv, val.csv, test.csv
run_name: resumefit_distilbert_v1
output_dir: /kaggle/working/runs

model_name: distilbert-base-uncased
resume_max_tokens: 350   # resumes front-load the most relevant info (summary, recent experience)
jd_max_tokens: 150       # so a fixed per-side budget beats naive concatenated truncation
max_seq_len: 512

batch_size: 16
num_workers: 2
epochs: 4

lr: 0.00002
warmup_ratio: 0.1
weight_decay: 0.01

early_stopping_patience: 2
seed: 42


In [ ]:
%%writefile /kaggle/working/train.py
"""Fine-tune DistilBERT as a 3-class resume<->job-description fit classifier.

Single-phase full fine-tune (unlike the image projects' freeze/unfreeze recipe --
transformer fine-tuning conventionally trains all layers at a low LR from the start).
Class-weighted loss handles the label imbalance (~50% No Fit / 25% / 25%).

Usage:
    python train.py --config ../configs/train_config.yaml
"""
from __future__ import annotations

import argparse
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yaml
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, get_linear_schedule_with_warmup

LABELS = ["No Fit", "Potential Fit", "Good Fit"]


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


MIN_SUPPORTED_CUDA_CAPABILITY = (7, 0)  # older GPUs (e.g. Kaggle's P100, sm_60) aren't supported
# by recent PyTorch prebuilt wheels' compiled kernels; Kaggle's GPU pool mixes P100 and T4 and
# doesn't let you pin which one you get, so detect and fall back instead of crashing.


def select_device() -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    capability = torch.cuda.get_device_capability(0)
    if capability < MIN_SUPPORTED_CUDA_CAPABILITY:
        name = torch.cuda.get_device_name(0)
        print(f"GPU {name} has CUDA capability {capability}, below the minimum "
              f"{MIN_SUPPORTED_CUDA_CAPABILITY} this PyTorch build supports -- falling back to CPU")
        return torch.device("cpu")
    return torch.device("cuda")


class ResumeFitDataset(Dataset):
    """Tokenizes resume and JD separately with fixed per-side token budgets, then
    combines as [CLS] resume [SEP] jd [SEP] -- avoids the default pair-truncation
    strategy silently favoring whichever text happens to be shorter.
    """

    def __init__(self, df: pd.DataFrame, tokenizer, resume_max_tokens: int, jd_max_tokens: int):
        self.resumes = df["resume_text"].tolist()
        self.jds = df["job_description_text"].tolist()
        self.labels = [LABELS.index(label) for label in df["label"]]
        self.tokenizer = tokenizer
        self.resume_max_tokens = resume_max_tokens
        self.jd_max_tokens = jd_max_tokens

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        resume_ids = self.tokenizer.encode(
            self.resumes[idx], add_special_tokens=False, truncation=True, max_length=self.resume_max_tokens
        )
        jd_ids = self.tokenizer.encode(
            self.jds[idx], add_special_tokens=False, truncation=True, max_length=self.jd_max_tokens
        )
        input_ids = [self.tokenizer.cls_token_id] + resume_ids + [self.tokenizer.sep_token_id] + \
            jd_ids + [self.tokenizer.sep_token_id]
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def collate(batch, pad_token_id: int):
    max_len = max(len(item["input_ids"]) for item in batch)
    input_ids = torch.full((len(batch), max_len), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
    labels = torch.stack([item["label"] for item in batch])
    for i, item in enumerate(batch):
        n = len(item["input_ids"])
        input_ids[i, :n] = item["input_ids"]
        attention_mask[i, :n] = 1
    return {"input_ids": input_ids, "attention_mask": attention_mask, "label": labels}


def run_epoch(model, loader, criterion, optimizer, scheduler, device, train: bool, log_every: int = 25):
    model.train(mode=train)
    total_loss, n = 0.0, 0
    all_labels, all_preds = [], []
    phase = "train" if train else "val"
    start = time.time()
    for step, batch in enumerate(loader, start=1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        with torch.set_grad_enabled(train):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()
        batch_size = input_ids.size(0)
        total_loss += loss.item() * batch_size
        n += batch_size
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())

        if step % log_every == 0 or step == len(loader):
            elapsed = time.time() - start
            print(f"  [{phase}] batch {step}/{len(loader)} "
                  f"({elapsed:.1f}s elapsed, {elapsed / step:.2f}s/batch, running_loss={total_loss / n:.4f})",
                  flush=True)

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return total_loss / n, macro_f1


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True, type=Path)
    args = parser.parse_args()

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    set_seed(cfg["seed"])
    device = select_device()
    print(f"device: {device}")

    data_dir = Path(cfg["data_dir"])
    train_df = pd.read_csv(data_dir / "train.csv")
    val_df = pd.read_csv(data_dir / "val.csv")

    tokenizer = DistilBertTokenizerFast.from_pretrained(cfg["model_name"])
    train_ds = ResumeFitDataset(train_df, tokenizer, cfg["resume_max_tokens"], cfg["jd_max_tokens"])
    val_ds = ResumeFitDataset(val_df, tokenizer, cfg["resume_max_tokens"], cfg["jd_max_tokens"])

    collate_fn = lambda batch: collate(batch, tokenizer.pad_token_id)
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                               num_workers=cfg["num_workers"], collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False,
                             num_workers=cfg["num_workers"], collate_fn=collate_fn)

    class_weights = compute_class_weight("balanced", classes=np.arange(len(LABELS)), y=train_ds.labels)
    print(f"class weights: {dict(zip(LABELS, class_weights))}")
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))

    model = DistilBertForSequenceClassification.from_pretrained(cfg["model_name"], num_labels=len(LABELS)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    total_steps = len(train_loader) * cfg["epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * cfg["warmup_ratio"]), num_training_steps=total_steps
    )

    run_dir = Path(cfg["output_dir"]) / cfg["run_name"]
    ckpt_dir = run_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    with open(run_dir / "labels.json", "w") as f:
        json.dump(LABELS, f, indent=2)

    # Auto-resume: a Kaggle CPU run can take hours and risks hitting the platform's session
    # time limit, so every epoch's full training state (not just the best model weights) is
    # checkpointed -- rerunning the same command picks up where it left off instead of
    # restarting from scratch.
    state_path = ckpt_dir / "last_state.pt"
    if state_path.exists():
        state = torch.load(state_path, map_location=device)
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        scheduler.load_state_dict(state["scheduler_state_dict"])
        start_epoch = state["epoch"] + 1
        best_macro_f1 = state["best_macro_f1"]
        epochs_without_improvement = state["epochs_without_improvement"]
        history = state["history"]
        print(f"resuming from {state_path}: starting at epoch {start_epoch} "
              f"(best_val_macro_f1={best_macro_f1:.4f} so far)")
    else:
        start_epoch = 1
        best_macro_f1 = -1.0
        epochs_without_improvement = 0
        history = []

    for epoch in range(start_epoch, cfg["epochs"] + 1):
        train_loss, train_f1 = run_epoch(model, train_loader, criterion, optimizer, scheduler, device, train=True)
        val_loss, val_f1 = run_epoch(model, val_loader, criterion, optimizer, scheduler, device, train=False)
        print(f"epoch {epoch}: train_loss={train_loss:.4f} train_macro_f1={train_f1:.4f} "
              f"val_loss={val_loss:.4f} val_macro_f1={val_f1:.4f}")
        history.append({"epoch": epoch, "train_loss": train_loss, "train_macro_f1": train_f1,
                         "val_loss": val_loss, "val_macro_f1": val_f1})

        if val_f1 > best_macro_f1:
            best_macro_f1 = val_f1
            epochs_without_improvement = 0
            model.save_pretrained(ckpt_dir / "best")
            tokenizer.save_pretrained(ckpt_dir / "best")
        else:
            epochs_without_improvement += 1

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_macro_f1": best_macro_f1,
            "epochs_without_improvement": epochs_without_improvement,
            "history": history,
        }, state_path)

        if epochs_without_improvement >= cfg["early_stopping_patience"]:
            print(f"early stopping at epoch {epoch} (best val_macro_f1={best_macro_f1:.4f})")
            break

    with open(run_dir / "metrics.json", "w") as f:
        json.dump({"history": history, "best_val_macro_f1": best_macro_f1}, f, indent=2)

    print(f"done. best val_macro_f1={best_macro_f1:.4f}. best checkpoint: {ckpt_dir / 'best'}")


if __name__ == "__main__":
    main()


In [ ]:
!python /kaggle/working/train.py --config /kaggle/working/train_config.yaml

## 3. Evaluate on the held-out test split

Headline metric is **macro-F1** (fairer than accuracy given the ~50/25/25 label imbalance).

In [ ]:
%%writefile /kaggle/working/evaluate.py
"""Evaluate a trained resume-fit checkpoint against the held-out test split.

Usage:
    python evaluate.py --config ../configs/train_config.yaml \
        --checkpoint /kaggle/working/runs/resumefit_distilbert_v1/checkpoints/best
"""
from __future__ import annotations

import argparse
from pathlib import Path

import pandas as pd
import torch
import yaml
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

from train import LABELS, ResumeFitDataset, collate, select_device


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True, type=Path)
    parser.add_argument("--checkpoint", required=True, type=Path)
    parser.add_argument("--split", default="test", choices=["val", "test"])
    args = parser.parse_args()

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    device = select_device()

    tokenizer = DistilBertTokenizerFast.from_pretrained(args.checkpoint)
    model = DistilBertForSequenceClassification.from_pretrained(args.checkpoint).to(device).eval()

    df = pd.read_csv(Path(cfg["data_dir"]) / f"{args.split}.csv")
    ds = ResumeFitDataset(df, tokenizer, cfg["resume_max_tokens"], cfg["jd_max_tokens"])
    collate_fn = lambda batch: collate(batch, tokenizer.pad_token_id)
    loader = torch.utils.data.DataLoader(ds, batch_size=cfg["batch_size"], shuffle=False, collate_fn=collate_fn)

    all_labels, all_preds = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(input_ids=batch["input_ids"].to(device),
                            attention_mask=batch["attention_mask"].to(device)).logits
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(batch["label"].numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    print(f"split={args.split} n={len(all_labels)} macro_f1={macro_f1:.4f}")
    print(classification_report(all_labels, all_preds, target_names=LABELS, zero_division=0))
    print("confusion matrix (rows=true, cols=pred):")
    print(pd.DataFrame(confusion_matrix(all_labels, all_preds), index=LABELS, columns=LABELS))


if __name__ == "__main__":
    main()


In [ ]:
!python /kaggle/working/evaluate.py --config /kaggle/working/train_config.yaml     --checkpoint /kaggle/working/runs/resumefit_distilbert_v1/checkpoints/best --split test

## 4. Export to ONNX

Exports the model + a standalone `tokenizer.json` (loadable via the lightweight `tokenizers` package, no `transformers`/`torch` needed at inference), and verifies the ONNX output matches PyTorch within tolerance before trusting the artifact.

In [ ]:
%%writefile /kaggle/working/export_model.py
"""Export a trained resume-fit checkpoint to ONNX for CPU inference, quantize it to int8,
and verify both steps.

The fp32 export is ~255MB -- over GitHub's 100MB hard push limit -- so int8 dynamic
quantization (~67MB) is a required step here, not an optional optimization. Also saves a
standalone tokenizer.json so the Streamlit app can tokenize with the lightweight
`tokenizers` package instead of pulling in full `transformers` + `torch`.

Usage:
    python export_model.py \
        --checkpoint /kaggle/working/runs/resumefit_distilbert_v1/checkpoints/best \
        --output ../../models/resume_fit_distilbert.onnx
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import onnxruntime as ort
import torch
from onnxruntime.quantization import QuantType, quantize_dynamic
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

LABELS = ["No Fit", "Potential Fit", "Good Fit"]


def softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True, type=Path)
    parser.add_argument("--output", required=True, type=Path)
    args = parser.parse_args()

    tokenizer = DistilBertTokenizerFast.from_pretrained(args.checkpoint)
    model = DistilBertForSequenceClassification.from_pretrained(args.checkpoint)
    model.eval()

    dummy_len = 32
    dummy_input_ids = torch.randint(0, tokenizer.vocab_size, (1, dummy_len), dtype=torch.long)
    dummy_attention_mask = torch.ones((1, dummy_len), dtype=torch.long)

    args.output.parent.mkdir(parents=True, exist_ok=True)
    fp32_path = args.output.with_suffix(".fp32.onnx")
    torch.onnx.export(
        model,
        (dummy_input_ids, dummy_attention_mask),
        str(fp32_path),
        input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch", 1: "seq"},
            "attention_mask": {0: "batch", 1: "seq"},
            "logits": {0: "batch"},
        },
        opset_version=17,
        dynamo=False,  # legacy TorchScript-based exporter; avoids the onnxscript/onnx_ir dependency
    )
    print(f"exported to {fp32_path}")

    with torch.no_grad():
        torch_out = model(input_ids=dummy_input_ids, attention_mask=dummy_attention_mask).logits.numpy()

    sess = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
    onnx_out = sess.run(["logits"], {
        "input_ids": dummy_input_ids.numpy(),
        "attention_mask": dummy_attention_mask.numpy(),
    })[0]

    max_diff = np.abs(torch_out - onnx_out).max()
    print(f"max abs diff in logits (torch vs onnx): {max_diff:.6f}")

    torch_prob = softmax(torch_out)
    onnx_prob = softmax(onnx_out)
    max_prob_diff = np.abs(torch_prob - onnx_prob).max()
    print(f"max abs diff in probability (torch vs onnx): {max_prob_diff:.6f}")
    assert max_prob_diff < 0.01, "ONNX export does not match PyTorch output — do not trust this artifact"
    print("fp32 export verified OK")

    quantize_dynamic(str(fp32_path), str(args.output), weight_type=QuantType.QInt8)
    fp32_size = fp32_path.stat().st_size / 1024 / 1024
    quant_size = args.output.stat().st_size / 1024 / 1024
    fp32_path.unlink()
    print(f"quantized {fp32_size:.1f}MB -> {quant_size:.1f}MB: {args.output}")

    quant_sess = ort.InferenceSession(str(args.output), providers=["CPUExecutionProvider"])
    quant_out = quant_sess.run(["logits"], {
        "input_ids": dummy_input_ids.numpy(),
        "attention_mask": dummy_attention_mask.numpy(),
    })[0]
    quant_prob_diff = np.abs(torch_prob - softmax(quant_out)).max()
    print(f"max abs diff in probability (torch fp32 vs quantized): {quant_prob_diff:.4f}")
    # Loose tolerance -- quantization is lossy by design (expect a few points of accuracy
    # loss, verified separately via evaluate.py against real data), this just catches a
    # broken/garbage quantized export, not fine-grained accuracy regression.
    assert quant_prob_diff < 0.5, "quantized model diverges too much from fp32 -- do not trust this artifact"
    print("quantized export verified OK")

    # Standalone tokenizer.json (loadable via the `tokenizers` package, no `transformers` needed).
    tokenizer_dst = args.output.parent / "tokenizer.json"
    tokenizer.backend_tokenizer.save(str(tokenizer_dst))
    print(f"saved tokenizer to {tokenizer_dst}")

    labels_dst = args.output.parent / "labels.json"
    with open(labels_dst, "w") as f:
        json.dump(LABELS, f, indent=2)
    print(f"saved labels to {labels_dst}")


if __name__ == "__main__":
    main()


In [ ]:
!python /kaggle/working/export_model.py     --checkpoint /kaggle/working/runs/resumefit_distilbert_v1/checkpoints/best     --output /kaggle/working/export/resume_fit_distilbert.onnx

## 5. Package for download

Download `fitcheck_model.zip` from this notebook's **Output** tab, unzip it into `FitCheck/models/` locally (should contain `resume_fit_distilbert.onnx`, `tokenizer.json`, `labels.json`), then run `streamlit run app.py`.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/fitcheck_model", "zip", "/kaggle/working/export")
print("packaged: /kaggle/working/fitcheck_model.zip")